In [0]:
%pip install -q -r requirements.txt

In [0]:
from neo4j import GraphDatabase
from openai import OpenAI
import os
from dotenv import load_dotenv

In [0]:
# Load environment variables and get credentials
load_dotenv()
URI = os.getenv('NEO4J_URI')
USER = os.getenv('NEO4J_USER')
PASSWORD = os.getenv('NEO4J_PASSWORD')
BASE_URL = os.getenv('LLM_URL')
API_KEY = os.getenv('LLM_API_KEY')

# Connect to neo4j database
AUTH = (USER, PASSWORD)
driver = GraphDatabase.driver(URI, auth=AUTH)

# LLM connection
client = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY
)

In [0]:
def retrieve_all_knowledge():
    
    # Retrieve all the nodes and relationships from the database
    query = """
    MATCH (n)
    OPTIONAL MATCH (n)-[r]->(m)
    RETURN n.name AS start_node, type(r) AS relation_type, m.name AS end_node
    """
    
    context = []
        
    with driver.session() as session:
        result = session.run(query)
        
        for record in result:
            if record["relation_type"] and record["end_node"]:
                line = f"[{record['start_node']}] --({record['relation_type']})--> [{record['end_node']}]"
                context.append(line)
            # stand alone point without connections
            elif record["start_node"]:
                line = f"[{record['start_node']}]"
                context.append(line)
                
    # remove duplicates and combine all the text
    unique_lines = list(set(context))
    context_str = "The following is the complete knowledge graph data of vanadium redox flow battery:\n" + "\n".join(unique_lines)
    
    return context_str

print(retrieve_all_knowledge())

In [0]:
def ask_battery_expert(user_question):
    graph_context = retrieve_all_knowledge()
    
    # LLM should only use the graph context to answer the question
    prompt = f"""You are a vanadium redox flow battery expert, and you have the following knowledge graph context:{graph_context}. Please answer the following question BASED ON THE KNOWLEDGE GRAPH CONTEXT. You don't have to show the user the original nodes and relationships, just give your conclusions. If the question is not related to the knowledge graph context, please answer "I don't know". User question: {user_question}
    """
    
    # Call LLM
    response = client.chat.completions.create(
        model="nvidia/nemotron-3-super-120b-a12b:free",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

# test
question = 'What might be the root cause of high internal resistance and what will happen if the internal resistance is too high?'
print(ask_battery_expert(question))